# Ungraded Lab: Feature Engineering Lab

## Task 1: Initial Setup and Data Loading

<b>Steps:</b>

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import seaborn as sns
import boto3

# Load preprocessed data from S3
df = pd.read_csv('ticketwise_dataset.csv')

# Display initial information
print("Dataset shape:", df.shape)
print("\nFeatures available:", df.columns.tolist())

## Task 2: Time-Based Feature Engineering

<b>Steps:</b> 

In [ ]:
def create_temporal_features(df):
    """
    Create time-based features for ticket routing
    
    Parameters:
        df: DataFrame with submission_time column
    
    Returns:
        DataFrame with new temporal features
    """
    # Your code here to:
    # 1. Create features from submission_time
    #    - Hour of day
    #    - Day of week
    #    - Is business hours (8am-6pm)

    # 2. Calculate time-based statistics
    #    - Average resolution time by hour
    #    - Average resolution time by day
    #    - Peak hours flag     return df

## Task 3: Customer Context Features

<b>Steps:</b> 

In [ ]:
# Create features that capture customer characteristics and history
def create_customer_features(df):
    """
    Create features based on customer context and history
    
    Parameters:
        df: DataFrame with customer information
    
    Returns:
        DataFrame with new customer-related features
    """
    # Your code here:
    # 1. Create priority score combining:
    #    - is_vip
    #    - contract_value
    
    # 2. Calculate historical metrics:
    #    - Previous interactions impact
    
    # 3. Customer segments based on priority score created

    
    return df

# Apply the function and verify results
df = create_customer_features(df)
print("\nNew customer features created:")
print(df.filter(like='customer_').columns)

## Task 4: AI-Assisted Feature Generation

<b>Steps:</b> 

In [ ]:
# Use this prompt with your preferred AI assistant:
"""
Given these columns in a support ticket dataset:
- submission_time, channel, priority
- customer metrics (subscription_tier, contract_value)
- resolution metrics (resolution_time, previous_interactions)
- priority flags (is_urgent, is_vip, self_declared_p1)

Suggest complex feature combinations that could help route tickets effectively.
"""

def implement_ai_features(df):
    """
    Implement AI-suggested feature combinations
    
    Parameters:
        df: DataFrame with base features
    
    Returns:
        DataFrame with AI-suggested features
    """
    # Your code here:
    # 1. Implement AI-suggested features
    # 2. Document the rationale for each feature
    # 3. Validate feature values
    
    return df

# Apply AI-suggested features
df = implement_ai_features(df)

## Task 5: Feature Selection and Evaluation

<b>Steps:</b> 

In [ ]:
def evaluate_features(df, target_col='resolution_time'):
    """
    Evaluate feature importance for ticket routing
    
    Parameters:
        df: DataFrame with all features
        target_col: Column to predict
    
    Returns:
        DataFrame with feature importance scores
    """
    # Prepare feature matrix and target
    X = df.select_dtypes(include=['float64', 'int64']).drop(columns=[target_col])
    y = df[target_col]
    
    # Calculate feature importance using Random Forest Regressor
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, y)
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'feature': X.columns,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Visualize top features
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df.head(10), x='importance', y='feature')
    plt.title('Top 10 Features for Ticket Routing')
    plt.tight_layout()
    plt.show()
    
    return importance_df
	

# Select final feature set
def select_final_features(df, importance_df, threshold=0.01):
    """
    Select final feature set based on importance and correlations
    
    Parameters:
        df: DataFrame with all features
        importance_df: Feature importance scores
        threshold: Minimum importance score
    
    Returns:
        List of selected features
    """
    # Your code here:
    # 1. Filter by importance threshold
    # 2. Check for correlations
    # 3. Remove redundant features
    # 4. Document selection rationale
    
    return selected_features

# Apply evaluate_features and select_final_features functions
# Your code here


### Solution Code
Need a hand or are curious to compare your approach? Below is a complete solution you  can use as a reference. This is just one of many valid ways to solve the problem. Make sure to give it a try on your own first. Use this implementation to troubleshoot, learn new techniques, or confirm your logic. Keep experimenting and enjoy the process!

## Task 1: Initial Setup and Data Loading Solution Code
<b>Steps:</b> 

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import seaborn as sns
import boto3

# Load preprocessed data from S3
df = pd.read_csv('ticketwise_dataset.csv')

# Display initial information
print("Dataset shape:", df.shape)
print("\nFeatures available:", df.columns.tolist())

## Task 2: Time-Based Feature Engineering Solution Code

<b>Steps:</b> 

In [ ]:
def create_temporal_features(df):
    """
    Create time-based features for ticket routing
    
    Parameters:
        df: DataFrame with submission_time column
    
    Returns:
        DataFrame with new temporal features
    """
    # Ensure submission_time is datetime
    df['submission_time'] = pd.to_datetime(df['submission_time'])
    
    # 1. Create features from submission_time
    df['hour'] = df['submission_time'].dt.hour
    df['day_of_week'] = df['submission_time'].dt.dayofweek  # 0=Monday
    df['is_business_hours'] = df['hour'].between(8, 18).astype(int)
    
    # 2. Calculate simple time-based statistics
    # Average resolution time by hour
    if 'resolution_time' in df.columns:
        avg_by_hour = df.groupby('hour')['resolution_time'].transform('mean')
        df['avg_resolution_by_hour'] = avg_by_hour

        # Average resolution time by day of week
        avg_by_day = df.groupby('day_of_week')['resolution_time'].transform('mean')
        df['avg_resolution_by_day'] = avg_by_day

        # Flag peak hours: hours where average resolution time is above overall mean
        overall_mean = df['resolution_time'].mean()
        df['peak_hour'] = (df['avg_resolution_by_hour'] > overall_mean).astype(int)
    
    return df

df = create_temporal_features(df)

## Task 3: Customer Context Features Solution Code
<b>Steps:</b> 

In [ ]:
# Create features that capture customer characteristics and history
def create_customer_features(df):
    """
    Create features based on customer context and history.

    Parameters:
        df: DataFrame with customer information
    
    Returns:
        DataFrame with new customer-related features

    """
    # 1. Priority score: simple weighted sum of VIP, contract, and urgent tickets
    df['customer_priority_score'] = (
        df['is_vip'] * 2 +
        (df['contract_value'] / df['contract_value'].max()) +
        df['is_urgent']
    )
    
    # 2. Historical metrics
    df['customer_open_ticket_ratio'] = df['open_tickets'] / (df['linked_ticket_count'] + 1)  # avoid division by zero
    
    # 3. Customer segments based on priority score
    df['customer_segment'] = pd.cut(
        df['customer_priority_score'],
        bins=[-float('inf'), 2, 4, float('inf')],
        labels=['Low', 'Medium', 'High']
    )
    
    return df


# Apply the function and verify results
df = create_customer_features(df)
print("\nNew customer features created:")
print(df.filter(like='customer_').columns)

## Task 4: AI-Assisted Feature Generation Solution Code
<b>Steps:</b> 

In [ ]:
# Use this prompt with your preferred AI assistant:
"""
Given these columns in a support ticket dataset:
- submission_time, channel, priority
- customer metrics (subscription_tier, contract_value)
- resolution metrics (resolution_time, previous_interactions)
- priority flags (is_urgent, is_vip, self_declared_p1)

Suggest complex feature combinations that could help route tickets effectively.
"""

def implement_ai_features(df):
    """
    Implement AI-suggested feature combinations
    
    Parameters:
        df: DataFrame with base features
    
    Returns:
        DataFrame with AI-suggested features
    """
    import pandas as pd

def implement_ai_features(df):
    """
    Implement AI-suggested feature combinations for ticket routing,
    document rationale, and validate feature values.
    """
    # Ensure submission_time is datetime
    df['submission_time'] = pd.to_datetime(df['submission_time'], errors='coerce')
    
    # 1. Feature creation
    
    # Customer importance score
    df['customer_importance'] = (
        df['is_vip'] * 2 +
        (df['contract_value'] / df['contract_value'].max()) +
        df['self_declared_p1']
    )
    
    # Urgency vs resolution time ratio
    df['urgency_resolve_ratio'] = df['is_urgent'] / (df['resolution_time'] + 1)
    
    # Experience-adjusted priority
    df['experience_adjusted_priority'] = df['customer_importance'] * (1 + df['previous_interactions'])
    
    # Submission hour and peak hours
    df['submission_hour'] = df['submission_time'].dt.hour
    df['is_peak_hour'] = df['submission_hour'].apply(lambda x: 1 if 9 <= x <= 17 else 0)
    
    # Combined routing score
    df['routing_score'] = (
        df['experience_adjusted_priority'] * 0.5 +
        df['urgency_resolve_ratio'] * 2 +
        df['is_peak_hour'] * 0.3
    )
    
    # 2. Document rationale
    feature_rationale = {
        'customer_importance': 'Important customers (VIP, high subscription, self-declared P1) should be prioritized.',
        'urgency_resolve_ratio': 'Tickets that are urgent but take longer to resolve need faster routing.',
        'experience_adjusted_priority': 'Accounts with multiple past interactions may indicate recurring issues.',
        'submission_hour': 'Hour of submission can show peak times needing faster attention.',
        'is_peak_hour': 'Tickets submitted during peak hours may require quicker routing.',
        'routing_score': 'Weighted combination of multiple features to determine ticket routing priority.'
    }
    
    print("\nFeature rationale:")
    for feat, rationale in feature_rationale.items():
        print(f"- {feat}: {rationale}")
    
    # 3. Validate feature values
    print("\nMissing values per new feature:")
    print(df[['customer_importance', 'urgency_resolve_ratio', 
              'experience_adjusted_priority', 'submission_hour', 
              'is_peak_hour', 'routing_score']].isna().sum())
    
    print("\nFeature value ranges:")
    print(df[['customer_importance', 'urgency_resolve_ratio', 
              'experience_adjusted_priority', 'routing_score']].agg(['min','max']))
    
    print("\nUnique values for peak hour indicator:")
    print(df['is_peak_hour'].unique())
 
    return df

# Apply AI-suggested features
df = implement_ai_features(df)

## Task 5: Feature Selection and Evaluation Solution Code
<b>Steps:</b> 

In [ ]:
def evaluate_features(df, target_col='resolution_time'):
    """
    Evaluate feature importance for ticket routing
    
    Parameters:
        df: DataFrame with all features
        target_col: Column to predict
    
    Returns:
        DataFrame with feature importance scores
    """
    # Prepare feature matrix and target
    X = df.select_dtypes(include=['float64', 'int64']).drop(columns=[target_col])
    y = df[target_col]
    
    # Calculate feature importance using Random Forest Regressor
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, y)
    
    # Create importance DataFrame
    importance_df = pd.DataFrame({
        'feature': X.columns,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Visualize top features
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df.head(10), x='importance', y='feature')
    plt.title('Top 10 Features for Ticket Routing')
    plt.tight_layout()
    plt.show()
    
    return importance_df
	
# Select final feature set
def select_final_features(df, importance_df, threshold=0.01):
    """
    Select final feature set based on importance and correlations
    
    Parameters:
        df: DataFrame with all features
        importance_df: Feature importance scores
        threshold: Minimum importance score
    
    Returns:
        List of selected features
    """
    # 1. Filter features by importance threshold
    selected_features = importance_df[importance_df['importance'] >= threshold]['feature'].tolist()
    
    # 2. Check correlations and remove redundant features
    corr_matrix = df[selected_features].corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    redundant_features = [col for col in upper_tri.columns if any(upper_tri[col] > 0.9)]
    
    # Remove highly correlated features
    selected_features = [f for f in selected_features if f not in redundant_features]
    
    # 3. Document rationale
    print("\nFeature selection rationale:")
    print(f"- Threshold importance >= {threshold}")
    print(f"- Removed highly correlated features (>0.9): {redundant_features}")
    
    return selected_features

# Apply evaluate_features and select_final_features functions
importance_df = evaluate_features(df, target_col='resolution_time')
selected_features = select_final_features(df, importance_df, threshold=0.01)
print("\nSelected features for modeling:")
print(selected_features)